In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
from IPython.display import Audio
from stable_baselines3 import PPO
# from sbx import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from sdlarch_rl.utils.utils import ExcludeButtonsWrapper, AugmentObservation, RandomStateWrapper
import sys
# from sbx.ppo.policies import CnnPolicy

import logging
logging.basicConfig(level=logging.DEBUG)


NUM_ENV = 6
SAVE_DIR="./model-sf4-all"
TENSORBOARD="./tensorboard-sf4-all"
TOTAL_TIMESTEP_NUMB = 50_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
EVAL_FREQU=CHECK_FREQ_NUMB*2
MAX_STEPS= 4_000
LEARNING_RATING=2e-4
#EVALS=15
EVALS=30 # all

#ENT_COEF = 0.001
ENT_COEF = 0.00001
n_steps=2048
# batch_size=64 * NUM_ENV
batch_size=128 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)

def linear_schedule(initial_lr):
    def schedule(progress_remaining):
        return progress_remaining * initial_lr
    return schedule

def make_env():
    def _init():
        env = make(
            "SuperStreetFighterIV-3DS",
            # render_mode="human"
        )

        # very easy
        # env = RandomStateWrapper(env, states=[
        #     'default', 
        #     'ryu_ken_very_easy',
        # ])

        # easy
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_easy', 
        #     'ryu_ken_easy_alternate',
        # ])

        # medium
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_medium_original', 
        #     'ryu_ken_medium_alternate',
        # ]) 

        # medium hard
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_medium_hard_original', 
        #     'ryu_ken_medium_hard_alternate',
        # ])

        # hard
        # env = RandomStateWrapper(env, states=[
        #     "ryu_ken_hard_original",
        #     "ryu_ken_hard_alternate",
        # ])

        # very hard
        # env = RandomStateWrapper(env, states=[
        #     "ryu_ken_very_hard_original",
        #     "ryu_ken_very_hard_alternate",
        # ])
        
        # hardest trainning
        # env = RandomStateWrapper(env, states=[
        #     'ryu_ken_hardest_original', 
        #     'ryu_ken_hardest_alternate',
        # ])

        # all levels
        env = RandomStateWrapper(env, states=[
            'default', 
            'ryu_ken_very_easy',

            'ryu_ken_easy', 
            'ryu_ken_easy_alternate',

            'ryu_ken_medium_original', 
            'ryu_ken_medium_alternate',

            'ryu_ken_medium_hard_original', 
            'ryu_ken_medium_hard_alternate',

            "ryu_ken_hard_original",
            "ryu_ken_hard_alternate",

            "ryu_ken_very_hard_original",
            "ryu_ken_very_hard_alternate",
            
            'ryu_ken_hardest_original', 
            'ryu_ken_hardest_alternate',
        ])

        buttons = env.unwrapped.buttons
        to_exclude = ["START", "SELECT", "L2", "R2", "L3", "R3", "A", "X", "Y"]
        
        env = ExcludeButtonsWrapper(env, buttons, to_exclude)

        env = AugmentObservation(env)
        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        if not hasattr(env, 'get_wrapper_attr'):
            env.get_wrapper_attr = lambda name: getattr(env, name)
        
        return env
    return _init

process_class = SubprocVecEnv
# if NUM_ENV == 1:
#     process_class = DummyVecEnv

env = make_vec_env(make_env(), n_envs=NUM_ENV, vec_env_cls=process_class)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

eval_env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
eval_env = VecFrameStack(eval_env, 4, channels_order='last')


def get_previous_best_reward(log_dir):
    """Better reward"""
    eval_logs_dir = Path(log_dir) / "eval_logs"
    
    if not eval_logs_dir.exists():
        print("Directory of eval_logs not founded")
        return -np.inf
    
    npz_files = list(eval_logs_dir.glob("*.npz"))
    if not npz_files:
        print("No .npz files found")
        return -np.inf
    
    best_mean_reward = -np.inf
    
    for npz_file in sorted(npz_files):
        try:
            data = np.load(npz_file, allow_pickle=True)
            
            if 'results' in data:
                results = data['results']
                
                for eval_idx in range(results.shape[0]):
                    episode_rewards = results[eval_idx]
                    
                    mean_reward = np.mean(episode_rewards)
                    
                    if mean_reward > best_mean_reward:
                        best_mean_reward = mean_reward
                        
        except Exception as e:
            print(f"Error on ready {npz_file.name}: {e}")
            continue
    
    if best_mean_reward > -np.inf:
        print(f"Better rewarnd mean (mean of n_eval_episodes episodes): {best_mean_reward:.4f}")
        return best_mean_reward
    
    return -np.inf

previous_best_reward = get_previous_best_reward(SAVE_DIR)

class PersistentEvalCallback(EvalCallback):
    """
    EvalCallback preserves the history and saves it after each evaluation.
    """
    def __init__(self, *args, initial_best=-np.inf, **kwargs):
        super().__init__(*args, **kwargs)
        
        self.initial_best = initial_best
        self.has_logged_goal = False
        self.log_file = Path(self.log_path + ".npz")
        
        if initial_best > -np.inf:
            self.best_mean_reward = initial_best
            print(f"\nEvalCallback using history data: {self.best_mean_reward:.4f}")

    def _on_step(self) -> bool:
        if not hasattr(self, 'best_mean_reward_initialized'):
            if self.initial_best > -np.inf:
                self.best_mean_reward = self.initial_best
            self.best_mean_reward_initialized = True

        if self.n_calls == 1 and self.initial_best > -np.inf and not self.has_logged_goal:
            print(f"Objetive: {self.initial_best:.4f}")
            self.has_logged_goal = True
        
        result = super()._on_step()
        
        if hasattr(self, 'last_eval_timestep') and self.model.num_timesteps > self.last_eval_timestep:
            self._save_evaluation_data()
        
        return result
    
    def _on_evaluation(self, locals_, globals_):
        """Call after any eval - stores the timestep"""
        print("_on_evaluation")
        super()._on_evaluation(locals_, globals_)
        self.last_eval_timestep = self.model.num_timesteps
    
    def _save_evaluation_data(self):
        """Save data from eval to file .npz"""
        print("_save_evaluation_data")
        if len(self.timesteps) == 0:
            return
        
        try:
            last_idx = -1
            new_timestep = self.timesteps[last_idx]
            new_result = self.results[last_idx]
            new_ep_length = self.ep_lengths[last_idx]
            
            if self.log_file.exists():
                existing_data = np.load(self.log_file, allow_pickle=True)
                
                if new_timestep in existing_data['timesteps']:
                    print(f"Eval timestep {new_timestep} already exist. Skipping.")
                    return
                
                combined_timesteps = np.concatenate([existing_data['timesteps'], [new_timestep]])
                combined_results = np.vstack([existing_data['results'], [new_result]])
                combined_ep_lengths = np.vstack([existing_data['ep_lengths'], [new_ep_length]])
                
            else:
                combined_timesteps = np.array([new_timestep])
                combined_results = np.array([new_result])
                combined_ep_lengths = np.array([new_ep_length])
            
            np.savez(
                self.log_file,
                timesteps=combined_timesteps,
                results=combined_results,
                ep_lengths=combined_ep_lengths
            )
            
            total = len(combined_timesteps)
            print(f"Saved: eval {total} (timestep: {new_timestep:,})")
            
            if len(self.timesteps) > 0:
                self.timesteps.pop(last_idx)
                self.results.pop(last_idx)
                self.ep_lengths.pop(last_idx)
            
        except Exception as e:
            print(f"Error on save eval: {e}")
            import traceback
            traceback.print_exc()

# class PersistentEvalCallback(EvalCallback):
#     """
#     EvalCallback, which preserves history between sessions.
#     """
#     def __init__(self, *args, initial_best=-np.inf, **kwargs):
#         super().__init__(*args, **kwargs)

#         self.initial_best = initial_best
#         self.has_logged_goal = False
        
#         self.existing_data_path = Path(self.log_path + ".npz") # / "evaluations.npz"
#         self.should_merge = self.existing_data_path.exists()
        
#         if self.should_merge:
#             print(f"File founded: {str(self.existing_data_path)}")
#             print(f"   New data will be appended")
#         else:
#             print(f"Error: no file founded: {str(self.existing_data_path)}")

#     def _on_step(self) -> bool:
#         if not hasattr(self, 'best_mean_reward_initialized'):
#             if self.initial_best > -np.inf:
#                 self.best_mean_reward = self.initial_best
#                 print(f"\nEvalCallback using better = {self.best_mean_reward:.4f}")
#             self.best_mean_reward_initialized = True

#         if self.n_calls == 1 and self.initial_best > -np.inf and not self.has_logged_goal:
#             print(f"Objective {self.initial_best:.4f}")
#             self.has_logged_goal = True
        
#         result = super()._on_step()
        
#         if hasattr(self, 'is_new_best') and self.is_new_best and self.initial_best > -np.inf:
#             improvement = self.best_mean_reward - self.initial_best
#             if improvement > 0:
#                 print(f"\nBetter! +{improvement:.4f}")
#                 print(f" {self.best_mean_reward:.4f} > {self.initial_best:.4f}")
        
#         return result
    
    # def _dump_logs(self) -> None:
    #     """Overriding existenting data"""
    #     if len(self.timesteps) == 0:
    #         return
        
    #     if self.should_merge and self.existing_data_path.exists():
    #         try:
    #             existing_data = np.load(self.existing_data_path, allow_pickle=True)
                
    #             combined_timesteps = np.concatenate([
    #                 existing_data['timesteps'], 
    #                 np.array(self.timesteps)
    #             ])
    #             combined_results = np.vstack([
    #                 existing_data['results'], 
    #                 np.array(self.results)
    #             ])
    #             combined_ep_lengths = np.vstack([
    #                 existing_data['ep_lengths'], 
    #                 np.array(self.ep_lengths)
    #             ])
                
    #             np.savez(
    #                 self.existing_data_path,
    #                 timesteps=combined_timesteps,
    #                 results=combined_results,
    #                 ep_lengths=combined_ep_lengths
    #             )
                
    #             print(f"Data saved (APPEND): {len(combined_timesteps)} total ratings")
                
    #             self.timesteps = []
    #             self.results = []
    #             self.ep_lengths = []
                

    #             self.should_merge = False
                
    #         except Exception as e:
    #             print(f"Error on make append: {e}")
    #             super()._dump_logs()
    #     else:
    #         super()._dump_logs()

# class SmartEvalCallback(EvalCallback):
#     def __init__(self, *args, initial_best=-np.inf, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.initial_best = initial_best
#         self.has_logged_goal = False
    
#     def _on_step(self) -> bool:
#         if not hasattr(self, 'best_mean_reward_initialized'):
#             if self.initial_best > -np.inf:
#                 self.best_mean_reward = self.initial_best
#                 print(f"\nEvalCallback using history: best = {self.best_mean_reward:.4f}")
#             self.best_mean_reward_initialized = True

#         if self.n_calls == 1 and self.initial_best > -np.inf and not self.has_logged_goal:
#             print(f"Objective {self.initial_best:.4f}")
#             self.has_logged_goal = True
        
#         result = super()._on_step()
        
#         if hasattr(self, 'is_new_best') and self.is_new_best and self.initial_best > -np.inf:
#             improvement = self.best_mean_reward - self.initial_best
#             if improvement > 0:
#                 print(f"\nBetter! +{improvement:.4f}")
#                 print(f"   {self.best_mean_reward:.4f} > {self.initial_best:.4f}")
        
#         return result

# eval_callback = EvalCallback(
#     eval_env,
#     best_model_save_path=SAVE_DIR,
#     log_path=str(SAVE_DIR / "eval_logs"),
#     eval_freq=EVAL_FREQU,
#     deterministic=True,
#     render=False,
#     n_eval_episodes=10
# )

# todo: remove workaround
# workaround_value = 4.50
workaround_value = 4.35

if previous_best_reward > workaround_value:
    print(f"Using better historical: {previous_best_reward:.4f}")
else:
    print(f"Using default historical workaround:{workaround_value}")
    previous_best_reward = workaround_value

eval_callback = PersistentEvalCallback(
# eval_callback = SmartEvalCallback(
    eval_env,
    best_model_save_path=SAVE_DIR,
    log_path=str(SAVE_DIR / "eval_logs"),
    eval_freq=EVAL_FREQU,
    deterministic=True,
    render=False,
    n_eval_episodes=EVALS,
    initial_best=previous_best_reward
)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
    # model = RecurrentPPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        learning_rate=linear_schedule(LEARNING_RATING),
        batch_size=batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    model = PPO("CnnPolicy", 
    # model = RecurrentPPO('CnnLstmPolicy',
        env, 
        verbose=0, 
        # policy_kwargs=policy_kwargs, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
        learning_rate=linear_schedule(LEARNING_RATING)
    )


checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
callback = CallbackList([checkpoint_callback, eval_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_sf4")

env.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
Better rewarnd mean (mean of n_eval_episodes episodes): 4.3584
Using better historical: 4.3584

EvalCallback using history data: 4.3584
Loading existent model: model-sf4-all\best_model_6395000


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'D:\\Python311\\Lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'D:\\Python311\\Lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to c

Objetive: 4.3584
